# 1x1-conv-channel-reshape — faded example 1: Complete the per-pixel matmul for a channel-reducing 1x1 conv

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `1x1-conv-channel-reshape`. The last cell reports your progress on the `CNN: 1x1 conv channel-reshape` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 1x1 conv channel-reshape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`1x1-conv-channel-reshape`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "1x1-conv-channel-reshape"
DD_SUBTOPIC = "CNN: 1x1 conv channel-reshape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A 1x1 `nn.Conv2d(C_in, C_out, 1)` is a per-pixel linear map: `out[:,:,h,w] = W @ x[:,:,h,w] + b`, where `W = conv.weight.view(C_out, C_in)`. No spatial mixing happens, so flattening every pixel into a row and doing one matmul reproduces the conv exactly.

## Faded exercise 1

### Faded — reproduce a channel-reducing 1x1 conv with a matmul

Implement `one_by_one_channel_reduce(x, conv)` for `conv = nn.Conv2d(6, 2, kernel_size=1)`. The weight reshape, the spatial flatten, and the fold-back are given. You must supply the actual per-pixel linear computation: matrix-multiply the flattened pixels by the reshaped weight and add the (possibly `None`) bias. The output must equal `conv(x)` to fp32 tolerance and have shape `(B, C_out, H, W)`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch.nn as nn

def one_by_one_channel_reduce(x, conv):
    OC, IC, _, _ = conv.weight.shape
    B, _, H, W = x.shape
    Wlin = conv.weight.view(OC, IC)
    x_flat = rearrange(x, 'b c h w -> (b h w) c')
    out_flat = x_flat @ Wlin.t()
    if conv.bias is not None:
        out_flat = out_flat + conv.bias
    return rearrange(out_flat, '(b h w) c -> b c h w', b=B, h=H, w=W)

t.manual_seed(0)
conv = nn.Conv2d(6, 2, kernel_size=1)
x = t.randn(2, 6, 4, 5)
out = one_by_one_channel_reduce(x, conv)
print(tuple(out.shape))


def _test():
    import torch.nn as nn
    t.manual_seed(0)
    conv = nn.Conv2d(6, 2, kernel_size=1)
    x = t.randn(2, 6, 4, 5)
    out = one_by_one_channel_reduce(x, conv)
    assert out.shape == (2, 2, 4, 5), out.shape
    assert t.allclose(out, conv(x), atol=1e-5)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn

def one_by_one_channel_reduce(x, conv):
    OC, IC, _, _ = conv.weight.shape
    B, _, H, W = x.shape
    Wlin = conv.weight.view(OC, IC)
    x_flat = rearrange(x, 'b c h w -> (b h w) c')
    out_flat = x_flat @ Wlin.t()
    if conv.bias is not None:
        out_flat = out_flat + conv.bias
    return rearrange(out_flat, '(b h w) c -> b c h w', b=B, h=H, w=W)

t.manual_seed(0)
conv = nn.Conv2d(6, 2, kernel_size=1)
x = t.randn(2, 6, 4, 5)
out = one_by_one_channel_reduce(x, conv)
print(tuple(out.shape))
```
</details>